In [1]:
import gc
import torch
import openai
import pandas as pd

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

### Load Data

- Downloaded from Arctic Shift
- r/cybersecurity (global geopolitical security, surveillance, intersection of CS and social issues)
- posts from 2024-01-01 to 2026-04-06 6:20 PM

In [2]:
posts_df = pd.read_json('r_cybersecurity_posts.jsonl', lines=True)
posts_df

,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,url_overridden_by_dest,crosspost_parent,crosspost_parent_list,poll_data,author_cakeday,location_lat,location_long,location_name,websocket_url,outbound_link
0,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,AutoModerator,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,ibeawuchi,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,PanicInTheKernel,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,Warm_Ad_7120,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,NaiveJello1914,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95232,NaN,[],False,NaN,NaN,False,Wild-Rest-3627,None,None,[],...,https://youtube.com/shorts/VYvwsLGi9RI?si=-00Z...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdw3te?...,{'url': 'https://out.reddit.com/t3_1sdw3te?url...
95233,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://arstechnica.com/security/2026/04/new-r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95234,NaN,[],False,NaN,NaN,False,Astofol760,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95235,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://www.scworld.com/news/axios-maintainers...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdwya8?...,{'url': 'https://out.reddit.com/t3_1sdwya8?url...


In [3]:
comments_df = pd.read_json('r_cybersecurity_comments.jsonl', lines=True)
comments_df

,_meta,all_awardings,approved_at_utc,approved_by,archived,associated_award,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,treatment_tags,unrepliable_reason,ups,user_reports,author_cakeday,editable,body_sha1,nest_level,profile_img,profile_over_18
0,{'retrieved_2nd_on': 1704196851},[],NaN,NaN,False,NaN,AutoModerator,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
1,"{'is_edited': True, 'retrieved_2nd_on': 170419...",[],NaN,NaN,False,NaN,Xzarkuun,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
2,{'retrieved_2nd_on': 1704196970},[],NaN,NaN,False,NaN,MSXzigerzh0,None,None,[],...,[],NaN,2,[],NaN,NaN,NaN,NaN,NaN,NaN
3,{'retrieved_2nd_on': 1704197151},[],NaN,NaN,False,NaN,jwrig,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
4,{'retrieved_2nd_on': 1704197214},[],NaN,NaN,False,NaN,me_z,#373c3f,None,[],...,[],NaN,5,[],NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695184,NaN,[],NaN,NaN,False,NaN,makeiteasy_24,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://styles.redditmedia.com/t5_h54npz/style...,0.0
695185,NaN,[],NaN,NaN,False,NaN,1littlenapoleon,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://styles.redditmedia.com/t5_ep18l/styles...,0.0
695186,NaN,[],NaN,NaN,False,NaN,Forward_Switch1015,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/f42d464c-d...,1.0
695187,NaN,[],NaN,NaN,False,NaN,LastFisherman373,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/3afc41ce-6...,0.0


#### Data Cleaning
- Remove posts and comments by known reddit bots
- Remove posts and comments that were marked as '\[removed\]' or '\[deleted\]'
- Remove posts and comments by '\[deleted\]' authors
- Remove posts and comments shorter than a sentence
- Removed posts and comments shorter than the length of an average sentence (~15-20 words)

In [4]:
drop_users = {'AutoModerator', 'RemindMeBot', '[deleted]', 'alara_zero', 'flairassistant'}
drop_content = {'[removed]', '[deleted]'}
drop_flairs = {'Career Questions & Discussion', 'Tutorial'}
drop_titles = {'[ Removed by moderator ]', '[ Removed by Reddit ]'}

Posts

In [5]:
clean_posts = posts_df[~posts_df['author'].isin(drop_users)].copy()
clean_posts = clean_posts[~clean_posts['link_flair_text'].isin(drop_flairs)]

clean_posts['selftext'] = clean_posts['selftext'].fillna('')
clean_posts = clean_posts[~clean_posts['selftext'].isin(drop_content)]

clean_posts['title'] = clean_posts['title'].fillna('')
clean_posts = clean_posts[~clean_posts['title'].isin(drop_titles)]

clean_posts

,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,url_overridden_by_dest,crosspost_parent,crosspost_parent_list,poll_data,author_cakeday,location_lat,location_long,location_name,websocket_url,outbound_link
2,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,PanicInTheKernel,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,Warm_Ad_7120,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,elitetek,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,Itchy_Sprinkles,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,zer0pRiME-X,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95227,NaN,[],False,NaN,NaN,False,Big-Engineering-9365,None,None,[],...,https://threatroad.substack.com/p/controls-sha...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95232,NaN,[],False,NaN,NaN,False,Wild-Rest-3627,None,None,[],...,https://youtube.com/shorts/VYvwsLGi9RI?si=-00Z...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdw3te?...,{'url': 'https://out.reddit.com/t3_1sdw3te?url...
95233,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://arstechnica.com/security/2026/04/new-r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95235,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://www.scworld.com/news/axios-maintainers...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdwya8?...,{'url': 'https://out.reddit.com/t3_1sdwya8?url...


Comments

In [6]:
clean_comments = comments_df[~comments_df['author'].isin(drop_users)].copy()

clean_comments['body'] = clean_comments['body'].fillna('')
clean_comments = clean_comments[~clean_comments['body'].isin(drop_content)]

# remove all comments from dropped posts
valid_post_ids = "t3_" + clean_posts['id'].astype(str)
clean_comments = clean_comments[clean_comments['link_id'].isin(valid_post_ids)]

clean_comments

,_meta,all_awardings,approved_at_utc,approved_by,archived,associated_award,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,treatment_tags,unrepliable_reason,ups,user_reports,author_cakeday,editable,body_sha1,nest_level,profile_img,profile_over_18
29,{'retrieved_2nd_on': 1704199395},[],NaN,NaN,False,NaN,anevilbor,#373c3f,None,[],...,[],NaN,20,[],NaN,NaN,NaN,NaN,NaN,NaN
44,{'retrieved_2nd_on': 1704200329},[],NaN,NaN,False,NaN,MaxHedrome,None,None,[],...,[],NaN,10,[],NaN,NaN,NaN,NaN,NaN,NaN
73,{'retrieved_2nd_on': 1704202724},[],NaN,NaN,False,NaN,Upstairs_Lie_9379,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
78,{'retrieved_2nd_on': 1704202857},[],NaN,NaN,False,NaN,Shaaaaazam,None,None,[],...,[],NaN,4,[],NaN,NaN,NaN,NaN,NaN,NaN
99,{'retrieved_2nd_on': 1704205815},[],NaN,NaN,False,NaN,Immrsbdud,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695175,NaN,[],NaN,NaN,False,NaN,blipojones,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/97d190a3-8...,0.0
695177,NaN,[],NaN,NaN,False,NaN,blipojones,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/97d190a3-8...,0.0
695179,NaN,[],NaN,NaN,False,NaN,CammKelly,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/32d5d73b-c...,1.0
695181,NaN,[],NaN,NaN,False,NaN,todbatx,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/4a49f053-d...,0.0


In [7]:
p_df = clean_posts[['author', 'created_utc', 'id', 'score']].copy()
p_df['text'] = clean_posts['title'].fillna('') + ": " + clean_posts['selftext'].fillna('')
p_df['type'] = 'post'
p_df['link_id'] = "t3_" + p_df['id'].astype(str)

c_df = clean_comments[['author', 'created_utc', 'link_id', 'score']].copy()
c_df['text'] = clean_comments['body'].fillna('')
c_df['type'] = 'comment'

full_df = pd.concat([
    p_df[['author', 'text', 'created_utc', 'type', 'link_id', 'score']],
    c_df[['author', 'text', 'created_utc', 'type', 'link_id', 'score']]
], ignore_index=True)

full_df = full_df[full_df['text'].str.split().str.len() >= 15]
full_df['date'] = pd.to_datetime(full_df['created_utc'], unit='s').dt.strftime('%Y-%m-%d')
full_df.to_parquet('cybersec_full_data.parquet')

full_df

,author,text,created_utc,type,link_id,score,date
0,PanicInTheKernel,Canary tokens: I’m a fan of canarytokens.org a...,1704067769,post,t3_18vkn27,1,2024-01-01
1,Warm_Ad_7120,Evaluation of non-legitimate Scheduled Tasks p...,1704068145,post,t3_18vkr9q,1,2024-01-01
2,elitetek,"Penetration testing?: Hey guys, I’m about to p...",1704072783,post,t3_18vm4ym,1,2024-01-01
3,Itchy_Sprinkles,I'd like to become a soc analyst in the future...,1704072957,post,t3_18vm6ud,3,2024-01-01
4,zer0pRiME-X,Possibly the most sophisticated exploit ever: ...,1704078446,post,t3_18vnpac,1,2024-01-01
...,...,...,...,...,...,...,...
487097,Mysterious_Step1657,This is exactly the context that pure severity...,1775477828,comment,t3_1safq3a,1,2026-04-06
487098,blipojones,"Ye i'm looking to ""go local"", like IRL face to...",1775477923,comment,t3_1sdgj6n,1,2026-04-06
487099,blipojones,ye this is exactly the fuzzy area i was wonder...,1775478138,comment,t3_1sdgj6n,1,2026-04-06
487100,CammKelly,Yup. Until then its a really awkward discussio...,1775478356,comment,t3_1sd7rtm,1,2026-04-06


In [8]:
# clear out large variables
del posts_df, clean_posts, p_df
del comments_df, clean_comments, c_df
gc.collect()

24

### 1.1 Aggragate Stats

In [9]:
posts_only = full_df[full_df['type'] == 'post']
comments_only = full_df[full_df['type'] == 'comment']

postcount = len(posts_only)
commcount = len(comments_only)

print('Total Post Count (>=15 words) from 2024-01-01 to 2026-04-06:', postcount)
print('Total Comment Count (>=15 words) from 2024-01-01 to 2026-04-06:', commcount)

unique_users_count = full_df['author'].dropna().nunique()
print('Total Unique User Count from 2024-01-01 to 2026-04-06:', unique_users_count)

print()

print('Average Post Score from 2024-01-01 to 2026-04-06:', posts_only['score'].mean())
print('Median Post Score from 2024-01-01 to 2026-04-06:', posts_only['score'].median())
print('Maximum Post Score from 2024-01-01 to 2026-04-06:', posts_only['score'].max())

print()

print('Average Comment Score from 2024-01-01 to 2026-04-06:', comments_only['score'].mean())
print('Median Comment Score from 2024-01-01 to 2026-04-06:', comments_only['score'].median())
print('Maximum Comment Score from 2024-01-01 to 2026-04-06:', comments_only['score'].max())

Total Post Count (>=15 words) from 2024-01-01 to 2026-04-06: 25875
Total Comment Count (>=15 words) from 2024-01-01 to 2026-04-06: 297127
Total Unique User Count from 2024-01-01 to 2026-04-06: 69927

Average Post Score from 2024-01-01 to 2026-04-06: 25.237062801932368
Median Post Score from 2024-01-01 to 2026-04-06: 1.0
Maximum Post Score from 2024-01-01 to 2026-04-06: 8184

Average Comment Score from 2024-01-01 to 2026-04-06: 6.704658277436921
Median Comment Score from 2024-01-01 to 2026-04-06: 2.0
Maximum Comment Score from 2024-01-01 to 2026-04-06: 2343


### Prepare NLP Setup

- Refactor databases to something easier to parse for subsequent tasks
    - Create a merged column with post body + title in posts dataframe
    - Only preserve this column, along with author, creation_utc (date), and score columns
    - Repeat this with column database
    - Concatanate both into a full database
    
- Use BERTopic for assigning topics with its outlined [best practices](https://maartengr.github.io/BERTopic/getting_started/best_practices/best_practices.html)
    - Prevent stochastic behaviour (replicability)
    - Fix number of topics (10)
    - Use MTEB to find most appropriate embeddings for 2026
        - Mihaiii/Ivysaur is better than all-MiniLM-L6-v2 while having the same size
    - Use BERTopic's [integration with LLMs](https://maartengr.github.io/BERTopic/getting_started/representation/llm.html#truncating-documents) to generate topic labels and summaries.
        - Using Google's new gemma4:e2b for **Topic Labelling and Description** based on keywords and documents, locally using Ollama

In [10]:
# replicability
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', 
                  random_state=42) # random state is the main setting, the rest are defaults needed
                                   # for BERTopic

# topic count reduction (internal algo)
hdbscan_model = HDBSCAN(min_cluster_size=1000, # same idea as UMAP here
                        metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
# remove english stop words, look at bigrams for topics, + BERTopic default

client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')

prompt_label = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label (not more than 5 words, title case capitalization) in the following format:
topic: <topic label>
"""
label_model = OpenAI(client, model='gemma4:e2b', prompt=prompt_label,
                     generator_kwargs={"reasoning_effort" : "none"})

prompt_summary = """
I have a topic that is described by the following keywords: [KEYWORDS]
In this topic, the following documents are a small but representative subset of all documents in the topic:
[DOCUMENTS]

Based on the information above, please give a description of this topic in the following format:
topic: <description>
"""
summary_model = OpenAI(client, model='gemma4:e2b', prompt=prompt_summary, 
                       generator_kwargs={"reasoning_effort" : "none"})

representation_model = {
   "Label": label_model,
   "Description": summary_model,
}

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

embedding_model = SentenceTransformer("Mihaiii/Ivysaur", device=device)
embeddings = embedding_model.encode(full_df['text'].to_list(), show_progress_bar=True, batch_size=64)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,

    nr_topics=11, # one extra is for misc, will be dropped
    verbose=True
)

Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5047 [00:00<?, ?it/s]

In [11]:
topics, _ = topic_model.fit_transform(full_df['text'].to_list(), embeddings)

2026-04-29 02:59:49,679 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-29 03:02:55,192 - BERTopic - Dimensionality - Completed ✓
2026-04-29 03:02:55,195 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-29 03:04:19,775 - BERTopic - Cluster - Completed ✓
2026-04-29 03:04:19,783 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-29 03:04:40,428 - BERTopic - Representation - Completed ✓
2026-04-29 03:04:40,438 - BERTopic - Topic reduction - Reducing number of topics
2026-04-29 03:04:40,552 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 11/11 [00:49<00:00,  4.53s/it]
2026-04-29 03:06:18,353 - BERTopic - Representation - Completed ✓
2026-04-29 03:06:18,659 - BERTopic - Topic reduction - Reduced number of topics from 22 to 11


### 1.2 Topic Identification

In [12]:
topic_data = topic_model.get_topic_info()
total_convos = postcount + commcount

# formatting
topic_data = topic_data.drop(columns=['Name'])
topic_data = topic_data.drop([0]) # 0th row == -1 : topic for miscallaneous posts and comments as per BERTopic, ignore
topic_data['Share of Total (%)'] = (topic_data['Count']/total_convos)*100
topic_data = topic_data.rename(columns={'Representation':'Top (10) Keywords'})
topic_data = topic_data[['Topic', 'Label', 'Description', 'Count', 'Share of Total (%)', 'Top (10) Keywords', 'Representative_Docs']]


topic_data

,Topic,Label,Description,Count,Share of Total (%),Top (10) Keywords,Representative_Docs
1,0,[Cybersecurity Career Advice],[A comprehensive overview of the cybersecurity...,127843,39.579631,"[security, just, like, ai, don, work, people, ...",[This is FAR too big of a question. We're not ...
2,1,[Phishing Awareness Training],[The topic revolves around **phishing** attack...,11295,3.496882,"[email, phishing, emails, training, data, just...",[Well this statement is ridiculous. Are phishi...
3,2,[Password Security And Management],[The topic revolves around the practical aspec...,10748,3.327534,"[password, passwords, mfa, use, account, passw...",[Unfortunately security is not 100% guaranteed...
4,3,[Network Ip Bridging And Security],"[Network security, connectivity, and privacy i...",7512,2.325682,"[ip, vpn, firewall, network, traffic, browser,...",[That is one point: they don't have to.\n\n\nB...
5,4,[UN Cybercrime Convention],[International cooperation and legal framework...,3361,1.040551,"[china, russia, chinese, russian, government, ...","[""only a nation with formidable hacking chops,..."
6,5,[Political Power And Elections],[Political and electoral dynamics involving th...,2382,0.737457,"[trump, election, people, government, presiden...",[>You are contributing to the white-washing be...
7,6,[Ransomware Attack Prevention],[Ransomware attacks involve malicious assaults...,2185,0.676466,"[ransomware, ransom, data, pay, attack, backup...","[The term you are looking for is ""ransomware"" ..."
8,7,[Ciso Role And Strategy],"[The role, nature, and challenges of a Chief I...",1897,0.587303,"[ciso, cisos, security, business, role, risk, ...",[I am trying to hire a good ciso for a startup...
9,8,[Security Conferences And Events],"[Cybersecurity conferences and events, includi...",1061,0.328481,"[conference, conferences, bsides, defcon, rsa,...","[Big conferences, Blackhat EU and BSides Londo..."
10,9,[Zero Trust Security Principles],[Zero Trust (ZT) is a security framework cente...,1005,0.311144,"[zero trust, trust, zero, access, network, ide...",[You could start by implementing a Zero-trust ...


### Prepare Remaining Tasks

- Save relevant data
- Clear memory
- Restart notebook after this section

In [15]:
# this plot will be useful later for validating 1.3
topics_over_time = topic_model.topics_over_time(full_df['text'].to_list(), full_df['date'].to_list(), 
                                                nr_bins=24) # 2 years of data, 24 months
topic_model.visualize_topics_over_time(topics_over_time, topics=[i for i in range(0, 12)])

24it [13:52, 34.68s/it]


In [13]:
topic_model.visualize_heatmap()

In [16]:
full_df_labels = topic_model.get_document_info(full_df['text'].to_list())
full_df['Topic'] = full_df_labels['Topic']

full_df.to_parquet('cybersec_full_data.parquet')
topic_data.to_parquet('cybersec_topic_data.parquet')

In [17]:
del topic_model, embeddings, 
gc.collect()

300

### 1.3 Trending and Persistent Topics

- Organise data appropriately
    - Key Column: Months
    - Rows: Topics
    - Values: Count of occurences of a given topic for a given month

- Visualise using previous plot: Anything with one or more major spike is defined as a trending topic, the rest are persistent (since they must be persistent enough to be identified by BERTopic)

- Use [Coefficient of Variation](https://en.wikipedia.org/wiki/Coefficient_of_variation) to characterise the above empirically
    - Trending: CV > 0.7
    - Persistent: CV <= 0.7

In [ ]:
import pandas as pd

In [ ]:
full_df = pd.read_parquet('cybersec_full_data.parquet')
topic_data = pd.read_parquet('cybersec_topic_data.parquet')

full_df = full_df[full_df['Topic'] > -1] # -1: topic for miscallaneous posts and comments as per BERTopic, ignore

full_df['Month'] = pd.to_datetime(full_df['created_utc'],  unit='s').dt.to_period('M').dt.to_timestamp()
topic_timeline = full_df.groupby(['Month', 'Topic']).size().unstack(fill_value=0)

def classify_by_cv(series, threshold=0.7):
    if series.sum() == 0 or series.mean() == 0:
        return "insignificant"
    cv = series.std() / series.mean()
    if cv > threshold:
        return "Trending"
    else:
        return "Persistent"

cv_class = topic_timeline.apply(classify_by_cv)
topic_data['Over Time'] = topic_data['Topic'].map(cv_class)
topic_data.to_parquet('cybersec_topic_data.parquet')

topic_data[['Topic', 'Label', 'Over Time']]

,Topic,Label,Over Time
1,0,[CISO Strategy and Risk],Persistent
2,1,[cybersecurity career advice],Persistent
3,2,[US Russia Politics Ties],Trending
4,3,[AI Usage and Criticism],Persistent
5,4,[Appreciation and Feedback],Persistent
6,5,[cyber insurance risk],Persistent
7,6,[Darknet Security Podcasts],Persistent
8,7,[subreddit posting guidance],Persistent
9,8,[CTF Learning and Practice],Persistent
10,9,[social engineering attacks and syndrome],Persistent


### 1.4 Stance and Debate

#### Incorporating Feeback
- After discussing with Prof, I'm switching to doing stance analysis exclusively on non-news posts.
- Disagreements ended up matching with agreements, which is expected, given that communities like these largely agree on informational posts.
- So, to narrow things down, the Stance and Debate analysis will be done on posts that focus on r/cybersecurity users, which are far more likely to involve discourse. To do this we filter to keep data from posts that are exclusively from the following flairs:
    - Business Security Questions & Discussion
    - Career Questions & Discussion
    - Certification / Training Questions
    - Personal Support & Help!
    - Burnout / Leaving Cybersecurity
    - Starting Cybersecurity Career
- After this, we rerun the topic analysis on this subset of posts before proceeding with stance analysis

In [ ]:
posts_df = pd.read_json('r_cybersecurity_posts.jsonl', lines=True)
comments_df = pd.read_json('r_cybersecurity_comments.jsonl', lines=True)

drop_users = {'AutoModerator', 'RemindMeBot', '[deleted]', 'alara_zero', 'flairassistant'}
drop_content = {'[removed]', '[deleted]'}
drop_titles = {'[ Removed by moderator ]', '[ Removed by Reddit ]'}
keep_flairs = {'Business Security Questions & Discussion', 'Career Questions & Discussion','Certification / Training Questions', 
               'Personal Support & Help!', 'Burnout / Leaving Cybersecurity', 'Starting Cybersecurity Career'}

clean_posts = posts_df[~posts_df['author'].isin(drop_users)].copy()
clean_posts = clean_posts[clean_posts['link_flair_text'].isin(keep_flairs)]

clean_posts['selftext'] = clean_posts['selftext'].fillna('')
clean_posts = clean_posts[~clean_posts['selftext'].isin(drop_content)]

clean_posts['title'] = clean_posts['title'].fillna('')
clean_posts = clean_posts[~clean_posts['title'].isin(drop_titles)]

clean_comments = comments_df[~comments_df['author'].isin(drop_users)].copy()

clean_comments['body'] = clean_comments['body'].fillna('')
clean_comments = clean_comments[~clean_comments['body'].isin(drop_content)]

# remove all comments from posts with irrelevant flair
valid_post_ids = "t3_" + clean_posts['id'].astype(str)
clean_comments = clean_comments[clean_comments['link_id'].isin(valid_post_ids)]

p_df = clean_posts[['author', 'created_utc', 'id', 'score']].copy()
p_df['text'] = clean_posts['title'].fillna('') + ": " + clean_posts['selftext'].fillna('')
p_df['type'] = 'post'
p_df['link_id'] = "t3_" + p_df['id'].astype(str)

c_df = clean_comments[['author', 'created_utc', 'link_id', 'score']].copy()
c_df['text'] = clean_comments['body'].fillna('')
c_df['type'] = 'comment'

full_df = pd.concat([
    p_df[['author', 'text', 'created_utc', 'type', 'link_id', 'score']],
    c_df[['author', 'text', 'created_utc', 'type', 'link_id', 'score']]
], ignore_index=True)

full_df = full_df[full_df['text'].str.split().str.len() >= 15]
full_df['date'] = pd.to_datetime(full_df['created_utc'], unit='s').dt.strftime('%Y-%m-%d')
full_df.to_parquet('cybersec_discourse_full_data.parquet')

full_df

In [ ]:
del posts_df, clean_posts, p_df
del comments_df, clean_comments, c_df
gc.collect()

In [ ]:
# replicability
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', 
                  random_state=42) # random state is the main setting, the rest are defaults needed
                                   # for BERTopic

# topic count reduction (internal algo)
hdbscan_model = HDBSCAN(min_cluster_size=1000, # same idea as UMAP here
                        metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
# remove english stop words, look at bigrams for topics, + BERTopic default

client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')

prompt_label = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label (not more than 5 words, title case capitalization) in the following format:
topic: <topic label>
"""
label_model = OpenAI(client, model='gemma4:e2b', prompt=prompt_label,
                     generator_kwargs={"reasoning_effort" : "none"})

prompt_summary = """
I have a topic that is described by the following keywords: [KEYWORDS]
In this topic, the following documents are a small but representative subset of all documents in the topic:
[DOCUMENTS]

Based on the information above, please give a description of this topic in the following format:
topic: <description>
"""
summary_model = OpenAI(client, model='gemma4:e2b', prompt=prompt_summary, 
                       generator_kwargs={"reasoning_effort" : "none"})

representation_model = {
   "Label": label_model,
   "Description": summary_model,
}

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

embedding_model = SentenceTransformer("Mihaiii/Ivysaur", device=device)
embeddings = embedding_model.encode(full_df['text'].to_list(), show_progress_bar=True, batch_size=64)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,

    nr_topics=11, # one extra is for misc, will be dropped
    verbose=True
)

In [ ]:
# this plot will be useful later for validating 1.3
topics_over_time = topic_model.topics_over_time(full_df['text'].to_list(), full_df['date'].to_list(), 
                                                nr_bins=24) # 2 years of data, 24 months
topic_model.visualize_topics_over_time(topics_over_time, topics=[i for i in range(0, 22)])

In [ ]:
topics, _ = topic_model.fit_transform(full_df['text'].to_list(), embeddings)

In [ ]:
topic_data = topic_model.get_topic_info()
total_convos = postcount + commcount

# formatting
topic_data = topic_data.drop(columns=['Name'])
topic_data = topic_data.drop([0]) # 0th row == -1 : topic for miscallaneous posts and comments as per BERTopic, ignore
topic_data['Share of Total (%)'] = (topic_data['Count']/total_convos)*100
topic_data = topic_data.rename(columns={'Representation':'Top (10) Keywords'})
topic_data = topic_data[['Topic', 'Label', 'Description', 'Count', 'Share of Total (%)', 'Top (10) Keywords', 'Representative_Docs']]


topic_data

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
full_df_labels = topic_model.get_document_info(full_df['text'].to_list())
full_df['Topic'] = full_df_labels['Topic']

full_df.to_parquet('cybersec_discourse_full_data.parquet')
topic_data.to_parquet('cybersec_discourse_topic_data.parquet')

In [ ]:
del topic_model, embeddings, 
gc.collect()

130884

- Extract dominant position
    - For each topic, get the top 10 most upvoted posts/comments (score column)
    - Pass these to an LLM (gemma4:e2b) with an appropriate prompt to identify the dominant position

- Use zero-shot-classification with valhalla/distilbart-mnli-12-1 to classify 'favourable'/'opposed' labels for each post/comment

- Group by users and take the most frequent stance on each topic to categorise them.

- For each topic, extract 10 top rated posts/comments from each side, and extract arguments using the LLM

In [ ]:
import torch
import openai
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline

Prompts

In [ ]:
full_df = pd.read_parquet('cybersec_discourse_full_data.parquet')
topic_data = pd.read_parquet('cybersec_discourse_topic_data.parquet')

prompt_dominance = """
I have a topic labled as '[TOPIC]'. I have some sample documents for the topic:
[DOCUMENTS]

Based on the documents above, extract the short and succinct dominant position with respect to the topic in the following format:
topic: <dominant position>
"""

prompt_argument_fav = """
I have a topic labled as '[TOPIC]'. For this topic, I also have a stance: '[STANCE]'. I have some sample documents that favour the stance:
[DOCUMENTS]

Based on the information above, summarize the key arguments made by the documents in the following format:
stance: <summary>
"""

prompt_argument_opp = """
I have a topic labled as '[TOPIC]'. For this topic, I also have a stance: '[STANCE]'. I have some sample documents that oppose the stance:
[DOCUMENTS]

Based on the information above, summarize the key arguments made by the documents in the following format:
stance: <summary>
"""

client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')

def llm_generation(prompt_template, topic_label, documents, stance=None):
    docs_string = "\n".join([f"- {doc}" for doc in documents])
    prompt = prompt_template.replace("[TOPIC]", topic_label).replace("[DOCUMENTS]", docs_string)
    if stance:
        prompt = prompt.replace("[STANCE]", stance)
    
    #print(prompt)

    response = client.chat.completions.create(
        model="gemma4:e2b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    
    output = response.choices[0].message.content.strip()
    if ":" in output:
        return output.split(":", 1)[1].strip() 
        # grab everything after ':' and clear up whitespaces on either side
    return output

Dominant Stance

In [ ]:
topic_pos = []
for topic_id in tqdm(topic_data['Topic'].tolist()):

    topic_label = topic_data[topic_data['Topic'] == topic_id]['Label'].values[0][0]
    top_docs = full_df[full_df['Topic'] == topic_id].nlargest(10, 'score')['text'].tolist()

    #print(top_docs[0])

    dom_pos = llm_generation(prompt_dominance, topic_label, top_docs)
    topic_pos.append(dom_pos)

topic_data['Dominant Position'] = topic_pos
topic_data.to_parquet('cybersec_discourse_topic_data.parquet')

topic_data

  0%|          | 0/20 [00:00<?, ?it/s]

,Topic,Label,Description,Count,Share of Total (%),Top (10) Keywords,Representative_Docs,Over Time,Dominant Position
1,0,[CISO Strategy and Risk],"[Cybersecurity strategy, risk management, orga...",123782,25.353682,"[security, like, just, use, don, cybersecurity...",[Security Management and then CISO.\n\nDevelop...,Persistent,Insider threats and systemic risk management a...
2,1,[cybersecurity career advice],"[Cybersecurity career, job searching, professi...",57465,11.770284,"[just, like, people, job, don, know, good, wor...",[Yeah. The sad reality at times is that not al...,Persistent,Cybersecurity careers must contend with the re...
3,2,[US Russia Politics Ties],[The provided text explores the perceived and ...,8012,1.641060,"[china, russia, trump, chinese, russian, gover...","[Not in Russia, I know we are supposed to limi...",Trending,US Russia politics ties: US government actions...
4,3,[AI Usage and Criticism],"[The topic revolves around the use, impact, an...",4396,0.900412,"[ai, slop, just, like, use, people, use ai, th...","[Ai Ai, cagptain! , which ai ?, which ai was t...",Persistent,AI is frequently misused and applied superfici...
5,4,[Appreciation and Feedback],[expressions of gratitude and appreciation],2923,0.598704,"[thank, thanks, thank thank, feedback, advice,...","[Thank you!, thank you, Thank you thank you]",Persistent,acknowledgment of utility and practical applic...
6,5,[cyber insurance risk],[The topic revolves around the intersection of...,1448,0.296587,"[insurance, cyber insurance, cyber, healthcare...","[It’s a requirement for most cyber insurance, ...",Persistent,Cyber insurance is often inadequate for large ...
7,6,[Darknet Security Podcasts],[The topic revolves around a collection of con...,1213,0.248453,"[podcast, diaries, darknet, darknet diaries, p...","[Darknet Diaries ‼️, Is this a podcast?, What ...",Persistent,"A strong preference for narrative-driven, high..."
8,7,[subreddit posting guidance],[Guidance and recommendations for seeking assi...,816,0.167137,"[posting, posting guides, cybersecurity_help, ...","[Hi, we've analyzed your post with machine lea...",Persistent,expressive and attention-grabbing communication
9,8,[CTF Learning and Practice],"[Cybersecurity training and practice, focusing...",749,0.153414,"[ctf, ctfs, hackthebox, tryhackme, hack, hack ...","[Ctf meaning ?, I am interested in the CTF, A ...",Persistent,The learning and practice of CTFs is best appr...
10,9,[social engineering attacks and syndrome],[The topic revolves around the concept of soci...,656,0.134365,"[social engineering, social, engineering, synd...","[Social engineering., We call that social engi...",Persistent,Social engineering is the psychological mechan...


Stance Classification

In [ ]:
topic_data = pd.read_parquet('cybersec_discourse_topic_data.parquet')
full_df = pd.read_parquet('cybersec_discourse_full_data.parquet')

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-1", device=device)

full_df['Dominant Position Alignment'] = "NA"

threshold = 0.75
for _, row in tqdm(topic_data.iterrows(), total=len(topic_data), desc='Topic'):
    topic_id = row['Topic']
    dom_pos = row['Dominant Position']
    
    mask = (full_df['Topic'] == topic_id) #& (full_df['text'].str.len() >= 200)
    subset_indices = full_df[mask].index
    texts = full_df.loc[subset_indices, 'text'].tolist()
    
    if not texts:
        continue

    topic_stances = []
    batch_size = 128
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        results = classifier(
            batch_texts, 
            candidate_labels=["favourable", "opposed"], 
            hypothesis_template=f"The stance toward {dom_pos} is {{}}."
        )
        for res in results:
            top_label = res['labels'][0]
            top_score = res['scores'][0]
            
            if top_score >= threshold:
                topic_stances.append(top_label)
            else:
                topic_stances.append('NA')
    
    full_df.loc[subset_indices, 'Dominant Position Alignment'] = topic_stances

full_df.to_parquet('cybersec_discourse_full_data.parquet')
full_df

Using device: mps


Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

Topic:   0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/968 [00:00<?, ?it/s]

  0%|          | 0/449 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

,author,text,created_utc,type,parent_post_id,score,date,Topic,Dominant Position Alignment
0,PanicInTheKernel,Canary tokens: I’m a fan of canarytokens.org a...,1704067769,post,t3_18vkn27,1,2024-01-01,0,NA
1,Warm_Ad_7120,Evaluation of non-legitimate Scheduled Tasks p...,1704068145,post,t3_18vkr9q,1,2024-01-01,-1,NA
2,elitetek,"Penetration testing?: Hey guys, I’m about to p...",1704072783,post,t3_18vm4ym,1,2024-01-01,1,NA
3,Itchy_Sprinkles,I'd like to become a soc analyst in the future...,1704072957,post,t3_18vm6ud,3,2024-01-01,0,NA
4,zer0pRiME-X,Possibly the most sophisticated exploit ever: ...,1704078446,post,t3_18vnpac,1,2024-01-01,-1,NA
...,...,...,...,...,...,...,...,...,...
488216,blipojones,"Ye i'm looking to ""go local"", like IRL face to...",1775477923,comment,t3_1sdgj6n,1,2026-04-06,0,NA
488217,blipojones,ye this is exactly the fuzzy area i was wonder...,1775478138,comment,t3_1sdgj6n,1,2026-04-06,0,NA
488218,CammKelly,Yup. Until then its a really awkward discussio...,1775478356,comment,t3_1sd7rtm,1,2026-04-06,-1,NA
488219,todbatx,https://www.runzero.com/resources/deciphering-...,1775478536,comment,t3_1safq3a,1,2026-04-06,-1,NA


User Grouping

In [ ]:
topic_data = pd.read_parquet('cybersec_discourse_topic_data.parquet')
full_df = pd.read_parquet('cybersec_discourse_full_data.parquet')

user_topic_stances = full_df.groupby(['author', 'Topic', 'Dominant Position Alignment']).size().unstack(fill_value=0)

user_topic_stances['Dominant Position Alignment'] = user_topic_stances.idxmax(axis=1)

equal_mask = user_topic_stances['favourable'] == user_topic_stances['opposed']
user_topic_stances.loc[equal_mask, 'Dominant Position Alignment'] = 'NA'

user_topic_stances = user_topic_stances.reset_index()

stance_counts = user_topic_stances[
    user_topic_stances['Dominant Position Alignment'] != 'NA'].groupby(
        ['Topic', 'Dominant Position Alignment']).size().unstack(fill_value=0)

topic_data['Favourable User Count'] = topic_data['Topic'].map(stance_counts['favourable']).fillna(0).astype(int)
topic_data['Opposed User Count'] = topic_data['Topic'].map(stance_counts['opposed']).fillna(0).astype(int)

user_topic_stances.to_parquet('cybersec_discourse_user_data.parquet')
topic_data.to_parquet('cybersec_discourse_topic_data.parquet')

display(topic_data)
user_topic_stances

,Topic,Label,Description,Count,Share of Total (%),Top (10) Keywords,Representative_Docs,Over Time,Dominant Position,Favourable User Count,Opposed User Count
1,0,[CISO Strategy and Risk],"[Cybersecurity strategy, risk management, orga...",123782,25.353682,"[security, like, just, use, don, cybersecurity...",[Security Management and then CISO.\n\nDevelop...,Persistent,Insider threats and systemic risk management a...,498,147
2,1,[cybersecurity career advice],"[Cybersecurity career, job searching, professi...",57465,11.770284,"[just, like, people, job, don, know, good, wor...",[Yeah. The sad reality at times is that not al...,Persistent,Cybersecurity careers must contend with the re...,149,115
3,2,[US Russia Politics Ties],[The provided text explores the perceived and ...,8012,1.641060,"[china, russia, trump, chinese, russian, gover...","[Not in Russia, I know we are supposed to limi...",Trending,US Russia politics ties: US government actions...,93,34
4,3,[AI Usage and Criticism],"[The topic revolves around the use, impact, an...",4396,0.900412,"[ai, slop, just, like, use, people, use ai, th...","[Ai Ai, cagptain! , which ai ?, which ai was t...",Persistent,AI is frequently misused and applied superfici...,26,38
5,4,[Appreciation and Feedback],[expressions of gratitude and appreciation],2923,0.598704,"[thank, thanks, thank thank, feedback, advice,...","[Thank you!, thank you, Thank you thank you]",Persistent,acknowledgment of utility and practical applic...,2253,2
6,5,[cyber insurance risk],[The topic revolves around the intersection of...,1448,0.296587,"[insurance, cyber insurance, cyber, healthcare...","[It’s a requirement for most cyber insurance, ...",Persistent,Cyber insurance is often inadequate for large ...,17,7
7,6,[Darknet Security Podcasts],[The topic revolves around a collection of con...,1213,0.248453,"[podcast, diaries, darknet, darknet diaries, p...","[Darknet Diaries ‼️, Is this a podcast?, What ...",Persistent,"A strong preference for narrative-driven, high...",891,0
8,7,[subreddit posting guidance],[Guidance and recommendations for seeking assi...,816,0.167137,"[posting, posting guides, cybersecurity_help, ...","[Hi, we've analyzed your post with machine lea...",Persistent,expressive and attention-grabbing communication,549,0
9,8,[CTF Learning and Practice],"[Cybersecurity training and practice, focusing...",749,0.153414,"[ctf, ctfs, hackthebox, tryhackme, hack, hack ...","[Ctf meaning ?, I am interested in the CTF, A ...",Persistent,The learning and practice of CTFs is best appr...,18,2
10,9,[social engineering attacks and syndrome],[The topic revolves around the concept of soci...,656,0.134365,"[social engineering, social, engineering, synd...","[Social engineering., We call that social engi...",Persistent,Social engineering is the psychological mechan...,91,1


Dominant Position Alignment,author,Topic,NA,favourable,opposed,Dominant Position Alignment
0,------_---__-Sad,-1,3,0,0,NA
1,------_---__-Sad,0,2,0,0,NA
2,-----Redacted-----,-1,1,0,0,NA
3,-----_-_-_-_-_-----,0,1,0,0,NA
4,---0celot---,-1,12,0,0,NA
...,...,...,...,...,...,...
151605,zzztoken,0,2,0,0,NA
151606,zzztoken,1,2,0,0,NA
151607,zzztoken,3,1,0,0,NA
151608,zzzzeru,-1,2,0,0,NA


In [ ]:
import pandas as pd

topic_data = pd.read_parquet('cybersec_discourse_topic_data.parquet')
full_df = pd.read_parquet('cybersec_discourse_full_data.parquet')
user_df = pd.read_parquet('cybersec_discourse_user_data.parquet')

Argument Summary

In [ ]:
topic_data = pd.read_parquet('cybersec_topic_data.parquet')
full_df = pd.read_parquet('cybersec_full_data.parquet')

fav_summaries = []
opp_summaries = []

with tqdm(total=len(topic_data['Topic'].tolist())) as pbar:
    for topic_id in topic_data['Topic'].tolist():
        topic_label = topic_data[topic_data['Topic'] == topic_id]['Label'].values[0][0]
        dom_pos = topic_data[topic_data['Topic'] == topic_id]['Dominant Position'].values[0]
        
        topic_subset = full_df[full_df['Topic'] == topic_id]
        supp_docs = topic_subset[topic_subset['Dominant Position Alignment'] == 'favourable'].nlargest(10, 'score')['text'].tolist()
        opp_docs = topic_subset[topic_subset['Dominant Position Alignment'] == 'opposed'].nlargest(10, 'score')['text'].tolist()
        
        if supp_docs:
            supp_arg = llm_generation(prompt_argument_fav, topic_label, supp_docs, stance=f"{dom_pos}")
        else:
            supp_arg = "No significant supporting arguments found."
            
        if opp_docs:
            opp_arg = llm_generation(prompt_argument_opp, topic_label, opp_docs, stance=f"{dom_pos}")
        else:
            opp_arg = "No significant opposing arguments found."
            
        fav_summaries.append(supp_arg)
        opp_summaries.append(opp_arg)

        pbar.update(1)
        pbar.refresh()

topic_data['Arguments in Favour'] = fav_summaries
topic_data['Arguments in Opposition'] = opp_summaries

topic_data.to_parquet('cybersec_topic_data.parquet')
topic_data

  0%|          | 0/20 [00:00<?, ?it/s]

,Topic,Label,Description,Count,Share of Total (%),Top (10) Keywords,Representative_Docs,Over Time,Dominant Position,Favourable User Count,Opposed User Count,Arguments in Favour,Arguments in Opposition
1,0,[CISO Strategy and Risk],"[Cybersecurity strategy, risk management, orga...",123782,25.353682,"[security, like, just, use, don, cybersecurity...",[Security Management and then CISO.\n\nDevelop...,Persistent,Insider threats and systemic risk management a...,498,147,The documents emphasize the necessity of proac...,The documents express deep skepticism and dist...
2,1,[cybersecurity career advice],"[Cybersecurity career, job searching, professi...",57465,11.770284,"[just, like, people, job, don, know, good, wor...",[Yeah. The sad reality at times is that not al...,Persistent,Cybersecurity careers must contend with the re...,149,115,Cybersecurity careers must contend with the re...,The opposing documents express extreme frustra...
3,2,[US Russia Politics Ties],[The provided text explores the perceived and ...,8012,1.641060,"[china, russia, trump, chinese, russian, gover...","[Not in Russia, I know we are supposed to limi...",Trending,US Russia politics ties: US government actions...,93,34,The documents argue that the perception of Rus...,The documents argue that Russia is actively en...
4,3,[AI Usage and Criticism],"[The topic revolves around the use, impact, an...",4396,0.900412,"[ai, slop, just, like, use, people, use ai, th...","[Ai Ai, cagptain! , which ai ?, which ai was t...",Persistent,AI is frequently misused and applied superfici...,26,38,AI is frequently misused as a superficial solu...,The documents strongly support the idea that A...
5,4,[Appreciation and Feedback],[expressions of gratitude and appreciation],2923,0.598704,"[thank, thanks, thank thank, feedback, advice,...","[Thank you!, thank you, Thank you thank you]",Persistent,acknowledgment of utility and practical applic...,2253,2,Appreciation is most strongly expressed when t...,The provided feedback or advice is often perce...
6,5,[cyber insurance risk],[The topic revolves around the intersection of...,1448,0.296587,"[insurance, cyber insurance, cyber, healthcare...","[It’s a requirement for most cyber insurance, ...",Persistent,Cyber insurance is often inadequate for large ...,17,7,Cyber insurance is increasingly recognized as ...,The opposing arguments focus heavily on the et...
7,6,[Darknet Security Podcasts],[The topic revolves around a collection of con...,1213,0.248453,"[podcast, diaries, darknet, darknet diaries, p...","[Darknet Diaries ‼️, Is this a podcast?, What ...",Persistent,"A strong preference for narrative-driven, high...",891,0,The preference for cybersecurity podcasts is s...,No significant opposing arguments found.
8,7,[subreddit posting guidance],[Guidance and recommendations for seeking assi...,816,0.167137,"[posting, posting guides, cybersecurity_help, ...","[Hi, we've analyzed your post with machine lea...",Persistent,expressive and attention-grabbing communication,549,0,The documents argue that effective subreddit p...,No significant opposing arguments found.
9,8,[CTF Learning and Practice],"[Cybersecurity training and practice, focusing...",749,0.153414,"[ctf, ctfs, hackthebox, tryhackme, hack, hack ...","[Ctf meaning ?, I am interested in the CTF, A ...",Persistent,The learning and practice of CTFs is best appr...,18,2,The learning and practice of CTFs should be dr...,The value of CTF learning and practice is stro...
10,9,[social engineering attacks and syndrome],[The topic revolves around the concept of soci...,656,0.134365,"[social engineering, social, engineering, synd...","[Social engineering., We call that social engi...",Persistent,Social engineering is the psychological mechan...,91,1,Social engineering is a highly effective and i...,The opposing argument asserts that social engi...


https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter

In [ ]:
import chromadb
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
chroma_client = chromadb.PersistentClient(path="./cybersec_chroma_db")
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="Mihaiii/Ivysaur", device='mps'
)

collection = chroma_client.get_or_create_collection(
    name="cybersecurity_rag",
    embedding_function=sentence_transformer_ef
)
print(f"ChromaDB initialized. Current vectors in database: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB initialized. Current vectors in database: 164394


In [ ]:
df = pd.read_parquet("cybersec_full_data.parquet").sample(150000, random_state=42)

df

,author,text,created_utc,type,link_id,score,date
159260,Flowers169,"Perhaps, though I do know he was awarded the i...",1718093532,comment,t3_1dczap9,1,2024-06-11
467411,Wide_Feature4018,No. As a txt it cant. But if you converted to ...,1754790668,comment,t3_1mm5rqj,-1,2025-08-10
59851,Slawpy_Joe,"Yeah, and AWS cloud practitioner but that one ...",1705708699,comment,t3_19aa162,1,2024-01-19
278872,guardian416,"Verify that the file is actually deleted, look...",1731780031,comment,t3_1gs165j,1,2024-11-16
351991,sauvignonsucks,GDPR basically means we already can’t use anyt...,1740776718,comment,t3_1j0hjbq,-6,2025-02-28
...,...,...,...,...,...,...,...
155002,cavscout43,"""thought leaders"" have driven Linkedin as a pr...",1717598726,comment,t3_1d8hckp,36,2024-06-05
7144,Space_Fics,"Looking for a tool that checks the website ""l...",1715773948,post,t3_1csiwxq,3,2024-05-15
630505,boogieman22281,"I will, and honestly i might fail. But at leas...",1773761390,comment,t3_1rw71wb,1,2026-03-17
84604,timothymtorres,This is a similar mentality to low ballers on ...,1708794538,comment,t3_1aytewt,59,2024-02-24


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "?", "!", " ", ""]
)

batch_size = 2000
documents_batch = []
metadatas_batch = []
ids_batch = []

for index, row in tqdm(df.iterrows(), total=len(df), desc="Chunking and Embedding"):
    text = str(row['text'])

    chunks = text_splitter.split_text(text)
    
    for chunk_idx, chunk in enumerate(chunks):
        documents_batch.append(chunk)
        
        metadatas_batch.append({
            "link_id": row['link_id'],
            "type": row['type'],
            "author": row['author']
        })
        
        ids_batch.append(f"{index}_chunk_{chunk_idx}")
        
    if len(documents_batch) >= batch_size:
        collection.add(
            documents=documents_batch,
            metadatas=metadatas_batch,
            ids=ids_batch
        )
        documents_batch = []
        metadatas_batch = []
        ids_batch = []

if documents_batch:
    collection.add(
        documents=documents_batch,
        metadatas=metadatas_batch,
        ids=ids_batch
    )

print(f"Ingestion complete. Total vectors in DB: {collection.count()}")

Chunking and Embedding:   0%|          | 0/150000 [00:00<?, ?it/s]

Ingestion complete. Total vectors in DB: 164394


In [ ]:
def retrieve_with_posts(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    enriched_contexts = []
    
    for i in range(len(results['documents'][0])):
        doc_text = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        
        if meta['type'] == 'comment':
            target_link_id = meta['link_id']
            
            parent_post = df[(df['link_id'] == target_link_id) & (df['type'] == 'post')]
            
            if not parent_post.empty:
                parent_text = parent_post.iloc[0]['text']
                enriched_contexts.append(
                    f"RETRIEVED COMMENT/CHUNK: {doc_text}\n\nORIGINAL POST: {parent_text}"
                )
            else:
                enriched_contexts.append(f"RETRIEVED COMMENT/CHUNK: {doc_text}")
        else:
            # It's already a post, just use it
            enriched_contexts.append(f"RETRIEVED POST/CHUNK: {doc_text}")
            
    return "\n\n---\n\n".join(enriched_contexts)

In [ ]:
context = retrieve_with_posts("California Age Verification Law")
print(context)

RETRIEVED COMMENT/CHUNK: if that is true, then that's a really bad attempt.

i read the online safety act and even though I'm no lawyer I can guarantee that age verification based on viewing habits definitely won't satisfy the requirements.

---

RETRIEVED COMMENT/CHUNK: Yeah no, I could see the argument for age verification for some websites and software. I don't agree with it but I can understand the argument. This though? It's a device in my home, I own it.

---

RETRIEVED COMMENT/CHUNK: it's clickbait. I dug into the law and it's not age verification at all.  
It's SELF REPORTED AGE, that is passed down to apps (like instagram) so they can no longer lie in court that they didn't know how old you were and thats why they didn't obey the laws ALREADY IN PLACE.
